## NeuralAid u55_equivalent

**Mục tiêu:** NeuralAid u55_equivalent.

Paper/review là chuẩn; chi tiết chưa công bố được ghi là implementation assumption.

**Artifact sinh ra:** được ghi rõ trong code của cell này.


In [1]:
# Nhiệm vụ: cài Ethos-U Vela và xác nhận Kaggle gọi được Vela CLI.
import shutil
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ethos-u-vela", "pesq", "pystoi"],
    check=True,
)

VELA_BIN = shutil.which("vela")
assert VELA_BIN is not None, "Không tìm thấy Vela CLI sau khi cài ethos-u-vela."

version = subprocess.run(
    [VELA_BIN, "--version"],
    text=True,
    capture_output=True,
    check=True,
)
print("VELA_BIN:", VELA_BIN)
print(version.stdout or version.stderr)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.0 MB/s eta 0:00:00
VELA_BIN: /usr/local/bin/vela
5.1.0



## Environment và profile

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Environment và profile.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Environment và profile; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
import hashlib, json, math, random, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
MODEL_NAME = "neuralaid"
TRACK = "u55_equivalent"
RUN_PROFILE = "p100_8h_final"  # Set debug only for a smoke run.
SEED = 20260816
tf.keras.utils.set_random_seed(SEED)
PAPER_SAMPLES_PER_EPOCH = 20000
PROFILE = {
    "debug":{"paper_steps_per_epoch":2500,"batch":1,"chunks":24,"fp32_session_epochs":1,"fp32_steps_per_session_epoch":2,"qat_session_epochs":1,"qat_steps_per_session_epoch":1,"fp32_max_steps":2,"qat_max_steps":1,"lr_schedule_kind":"budget_normalized","warmup_steps":1,"session_limit_seconds":900,"fp32_deadline_seconds":300,"qat_deadline_seconds":600,"finalization_reserve_seconds":300,"evaluation_max_items":1,"final_in_session":True},
    "p100_2h_final":{"paper_steps_per_epoch":2500,"batch":8,"fp32_session_epochs":2,"fp32_steps_per_session_epoch":100,"qat_session_epochs":2,"qat_steps_per_session_epoch":20,"fp32_max_steps":200,"qat_max_steps":40,"lr_schedule_kind":"budget_normalized","warmup_steps":20,"session_limit_seconds":2*60*60,"fp32_deadline_seconds":75*60,"qat_deadline_seconds":100*60,"finalization_reserve_seconds":20*60,"evaluation_max_items":16,"final_in_session":True},
    "p100_4h_final":{"paper_steps_per_epoch":2500,"batch":8,"fp32_session_epochs":3,"fp32_steps_per_session_epoch":150,"qat_session_epochs":3,"qat_steps_per_session_epoch":40,"fp32_max_steps":450,"qat_max_steps":120,"lr_schedule_kind":"budget_normalized","warmup_steps":45,"session_limit_seconds":4*60*60,"fp32_deadline_seconds":160*60,"qat_deadline_seconds":205*60,"finalization_reserve_seconds":35*60,"evaluation_max_items":64,"final_in_session":True},
    "p100_8h_final":{"paper_steps_per_epoch":2500,"batch":8,"fp32_session_epochs":3,"fp32_steps_per_session_epoch":300,"qat_session_epochs":3,"qat_steps_per_session_epoch":100,"fp32_max_steps":900,"qat_max_steps":300,"lr_schedule_kind":"budget_normalized","warmup_steps":90,"session_limit_seconds":8*60*60,"fp32_deadline_seconds":4*60*60+45*60,"qat_deadline_seconds":6*60*60+40*60,"finalization_reserve_seconds":80*60,"evaluation_max_items":128,"final_in_session":True},
    "p100_longrun":{"paper_steps_per_epoch":2500,"batch":8,"fp32_session_epochs":200,"fp32_steps_per_session_epoch":2500,"qat_session_epochs":30,"qat_steps_per_session_epoch":500,"fp32_max_steps":500000,"qat_max_steps":15000,"lr_schedule_kind":"paper_epoch","warmup_steps":25000,"session_limit_seconds":12*60*60,"fp32_deadline_seconds":11*60*60+40*60,"qat_deadline_seconds":11*60*60+40*60,"finalization_reserve_seconds":20*60,"evaluation_max_items":None,"final_in_session":False},
    "p100_full_eval":{"paper_steps_per_epoch":2500,"batch":8,"fp32_session_epochs":0,"fp32_steps_per_session_epoch":0,"qat_session_epochs":0,"qat_steps_per_session_epoch":0,"fp32_max_steps":0,"qat_max_steps":0,"lr_schedule_kind":"none","warmup_steps":0,"session_limit_seconds":12*60*60,"fp32_deadline_seconds":0,"qat_deadline_seconds":0,"finalization_reserve_seconds":0,"evaluation_max_items":None,"final_in_session":False},
}
RUN_CONFIG = PROFILE[RUN_PROFILE]
SESSION_START = time.monotonic()
EXPERIMENT_ID = f"{MODEL_NAME}_{TRACK}_seed{SEED}"
RUN_DIR = Path("/kaggle/working") / EXPERIMENT_ID; RUN_DIR.mkdir(parents=True, exist_ok=True)
FULL_EVAL_CHECKPOINT = next(iter(Path("/kaggle/input").rglob("best_qat.weights.h5")), None)
if RUN_PROFILE == "p100_full_eval" and FULL_EVAL_CHECKPOINT is None: raise FileNotFoundError("attach a prior run Output containing best_qat.weights.h5")
VELA_ACCELERATOR = "ethos-u55-256"
GPU_TRAINING_REQUIRED = True
GPU_TRAIN_DEVICE = "/GPU:0"
def require_gpu_for_training():
    physical = tf.config.list_physical_devices("GPU")
    if GPU_TRAINING_REQUIRED and not physical:
        raise RuntimeError("CUDA GPU is required for NeuralAid train/QAT; enable Kaggle P100 accelerator.")
    if not physical: return {"required": False, "available": False}
    try: tf.config.experimental.set_memory_growth(physical[0], True)
    except RuntimeError: pass  # TensorFlow may already have initialized the Kaggle GPU.
    details = tf.config.experimental.get_device_details(physical[0])
    logical = tf.config.list_logical_devices("GPU")
    return {"required": True, "available": True, "device_target": GPU_TRAIN_DEVICE,
            "physical_name": physical[0].name, "logical_devices": [device.name for device in logical],
            "device_name": details.get("device_name", "unknown"),
            "compute_capability": list(details.get("compute_capability", ())),
            "tensorflow": tf.__version__, "cuda_build": tf.sysconfig.get_build_info().get("cuda_version"),
            "memory_policy": "growth_requested", "mixed_precision": "disabled", "xla": "disabled"}


## Dataset board-response

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Dataset board-response.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Dataset board-response; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
def resolve_dataset_root():
    required = {"train.csv", "val.csv", "test.csv"}
    candidates = []
    for candidate in Path("/kaggle/input").glob("*"):
        candidates.extend((candidate, candidate / "speech_enhancement_board_response_v1"))
    candidates.extend(Path("/kaggle/input").rglob("speech_enhancement_board_response_v1"))
    matches = [p for p in dict.fromkeys(candidates) if p.is_dir() and required.issubset({x.name for x in p.iterdir() if x.is_file()})]
    if len(matches) != 1: raise FileNotFoundError("dataset root candidates: " + ", ".join(map(str, matches)))
    return matches[0]
DATASET_ROOT = resolve_dataset_root()
DATASET_CONFIG = DATASET_ROOT / "dataset_config.json"
if not DATASET_CONFIG.is_file(): raise FileNotFoundError(f"missing Kaggle dataset_config.json: {DATASET_CONFIG}")
DATASET_PROVENANCE = json.loads(DATASET_CONFIG.read_text(encoding="utf-8"))
DATASET_PROVENANCE.update({"source_type": "board_response_custom", "paper_base_recipe": False, "reproduction_label": "custom_dataset_architecture_reproduction", "clip_seconds": DATASET_PROVENANCE["samples_per_clip"] / DATASET_PROVENANCE["sample_rate"]})
print("Custom board-response dataset: architecture/QAT reproduction, not paper data reproduction")
assert DATASET_PROVENANCE["sample_rate"] == 16000
def split(name, expected_count):
    frame = pd.read_csv(DATASET_ROOT / f"{name}.csv")
    required = {"mixture_id", "mixture_path", "clean_path"}
    if len(frame) != expected_count or not required.issubset(frame.columns): raise ValueError(f"invalid {name}.csv")
    return frame
train_pairs, val_pairs, test_pairs = split("train", 17500), split("val", 3750), split("test", 3750)


## Audio loader

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Audio loader.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Audio loader; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
SAMPLE_RATE = 16000
def read_audio(path):
    # Representative samples use DATASET_ROOT / rel and therefore arrive as pathlib.Path.
    # Preserve tf.data string tensors while normalizing native paths for tf.io.read_file.
    filename = path if tf.is_tensor(path) else tf.convert_to_tensor(str(path), dtype=tf.string)
    audio, rate = tf.audio.decode_wav(tf.io.read_file(filename), desired_channels=1)
    tf.debugging.assert_equal(rate, SAMPLE_RATE); tf.debugging.assert_rank(audio, 2)
    return tf.squeeze(audio, -1)


## DSP contract

**Mục tiêu:** Thực hiện riêng nhiệm vụ: DSP contract.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách DSP contract; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
# PAPER-CONFIRMED: Fs=16 kHz, [LB|LC|LF]=[96|96|64], FFT=256.
# IMPLEMENTATION ASSUMPTION: the paper gives the dual-window geometry but not
# a complete coefficient table.  The complementary profile below is therefore
# kept separately from the normalized deployment/reference reconstruction.
CHUNK_SAMPLES = 96
LOOKBACK_SAMPLES = 96
LOOKAHEAD_SAMPLES = 64
FFT_LENGTH = 256
FFT_BINS = FFT_LENGTH // 2 + 1
PRE_EMPHASIS = 0.97
PCM_FULL_SCALE = 32768.0
ANALYSIS_WINDOW = np.ones(FFT_LENGTH, dtype=np.float32)
SYNTHESIS_EDGE_GAIN = 1.0 / 3.0
# Paper order: [lookback | current | lookahead]. Drop lookback before OLA.
# The 160-sample synthesis window has complementary 64-sample overlap edges.
SYNTHESIS_WINDOW = np.concatenate([
    # Paper equation (after dropping the 6-ms lookback): 1/3, 1, 1/3.
    np.full(LOOKAHEAD_SAMPLES, SYNTHESIS_EDGE_GAIN, dtype=np.float32),
    np.ones(CHUNK_SAMPLES - LOOKAHEAD_SAMPLES, dtype=np.float32),
    np.full(LOOKAHEAD_SAMPLES, SYNTHESIS_EDGE_GAIN, dtype=np.float32),
])
assert FFT_LENGTH == LOOKBACK_SAMPLES + CHUNK_SAMPLES + LOOKAHEAD_SAMPLES
assert SYNTHESIS_WINDOW.size == CHUNK_SAMPLES + LOOKAHEAD_SAMPLES

def initial_preemphasis_state(batch_size):
    return tf.zeros([batch_size], dtype=tf.float32)

def pre_emphasis_stream(chunk, previous_sample):
    chunk = tf.cast(chunk, tf.float32)
    previous = tf.concat([previous_sample[:, None], chunk[:, :-1]], axis=1)
    return chunk - PRE_EMPHASIS * previous, chunk[:, -1]

def pre_emphasis_sequence(waveform):
    waveform = tf.cast(waveform, tf.float32)
    return waveform - PRE_EMPHASIS * tf.concat([tf.zeros_like(waveform[:, :1]), waveform[:, :-1]], axis=1)

def make_stream_contexts(waveform):
    waveform = tf.cast(waveform, tf.float32)
    length = tf.shape(waveform)[1]
    chunks = tf.cast(tf.math.ceil(tf.cast(length, tf.float32) / CHUNK_SAMPLES), tf.int32)
    target = chunks * CHUNK_SAMPLES
    preemphasized = pre_emphasis_sequence(waveform)
    preemphasized = tf.pad(preemphasized, [[0, 0], [0, target - length]])
    context = tf.pad(preemphasized, [[0, 0], [LOOKBACK_SAMPLES, LOOKAHEAD_SAMPLES]])
    return tf.signal.frame(context, FFT_LENGTH, CHUNK_SAMPLES, pad_end=False)

def frontend_frame(context):
    tf.debugging.assert_equal(tf.shape(context)[-1], FFT_LENGTH)
    return tf.signal.rfft(context * ANALYSIS_WINDOW, [FFT_LENGTH])

def pcm16_to_model_float(pcm): return tf.cast(pcm, tf.float32) / PCM_FULL_SCALE

def cmsis_rfft_packed_to_tf129x2(packed):
    """Reference layout adapter: DC/Nyquist real-only, bins 1..127 complex."""
    packed = tf.convert_to_tensor(packed, tf.float32); real = tf.concat([packed[..., :1], packed[..., 2::2], packed[..., 1:2]], -1)
    imag = tf.concat([tf.zeros_like(packed[..., :1]), packed[..., 3::2], tf.zeros_like(packed[..., :1])], -1)
    return tf.stack([real, imag], -1)

def tf129x2_to_cmsis_rfft_packed(tf_frame):
    real, imag = tf_frame[..., :, 0], tf_frame[..., :, 1]
    return tf.concat([real[..., :1], real[..., -1:], tf.reshape(tf.stack([real[..., 1:-1], imag[..., 1:-1]], -1), [tf.shape(real)[0], -1])], -1)

def paper_dual_window_synthesis(spectrum):
    """Literal operational interpretation: iRFFT, discard lookback, apply synthesis window."""
    time_frame = tf.signal.irfft(spectrum, [FFT_LENGTH])
    return time_frame[..., LOOKBACK_SAMPLES:] * SYNTHESIS_WINDOW

def normalized_reference_synthesis(spectrum):
    """Stable implementation adaptation used by the reference/deployment path."""
    return paper_dual_window_synthesis(spectrum)

def dual_window_istft(spectrum):
    """Backward-compatible name for normalized_reference_synthesis."""
    return normalized_reference_synthesis(spectrum)

def normalized_reference_ola(frames):
    weights = tf.broadcast_to(tf.constant(SYNTHESIS_WINDOW, frames.dtype), tf.shape(frames))
    signal = tf.signal.overlap_and_add(frames, CHUNK_SAMPLES)
    normalizer = tf.signal.overlap_and_add(weights, CHUNK_SAMPLES)
    return signal / tf.maximum(normalizer, tf.constant(1e-8, signal.dtype))

def corrected_ola(frames):
    return normalized_reference_ola(frames)

def initial_reconstruction_state(batch_size):
    # Cache #3 belongs to DSP reconstruction, never to an undocumented NN residual.
    return {"ola_tail": tf.zeros([batch_size, LOOKAHEAD_SAMPLES], tf.float32), "ola_weight_tail": tf.zeros([batch_size, LOOKAHEAD_SAMPLES], tf.float32)}

def synthesis_stream(short_frame, ola_tail, ola_weight_tail):
    """Emit one 96-sample chunk and preserve the 64-sample iSTFT/OLA tail."""
    head_weight = tf.constant(SYNTHESIS_WINDOW[:LOOKAHEAD_SAMPLES], short_frame.dtype) + ola_weight_tail
    head = (short_frame[:, :LOOKAHEAD_SAMPLES] + ola_tail) / head_weight
    body = short_frame[:, LOOKAHEAD_SAMPLES:CHUNK_SAMPLES] / tf.constant(SYNTHESIS_WINDOW[LOOKAHEAD_SAMPLES:CHUNK_SAMPLES], short_frame.dtype)
    return tf.concat([head, body], axis=1), short_frame[:, CHUNK_SAMPLES:], tf.constant(SYNTHESIS_WINDOW[CHUNK_SAMPLES:], short_frame.dtype)[None, :]


## Primitive recurrent cells

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Primitive recurrent cells.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Primitive recurrent cells; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
DUAL_PATH_BLOCKS = 6
MODEL_DIM = 32
HIDDEN_SIZE = 32
DECODER_TEMPORAL_KERNEL = 3
FREQUENCY_COMPRESSION = 4
COMPRESSED_BINS = math.ceil(FFT_BINS / FREQUENCY_COMPRESSION)

class PrimitiveGRU(tf.keras.layers.Layer):
    """Reference GRU with an active reset gate."""
    def __init__(self, units):
        super().__init__(); self.x_proj = tf.keras.layers.Dense(3 * units); self.h_proj = tf.keras.layers.Dense(3 * units, use_bias=False)
    def call(self, x, h):
        x_z, x_r, x_n = tf.split(qat_dense(self.x_proj, x), 3, axis=-1)
        h_z, h_r, h_n = tf.split(qat_dense(self.h_proj, h), 3, axis=-1)
        z, r = tf.sigmoid(x_z + h_z), tf.sigmoid(x_r + h_r)
        candidate = tf.tanh(x_n + r * h_n)
        h = z * h + (1.0 - z) * candidate
        return h, h

class U55PrimitiveGRU(tf.keras.layers.Layer):
    """Split-free GRU: algebraically identical to Dense(3H) followed by SPLIT."""
    def __init__(self, units):
        super().__init__()
        self.x_z, self.x_r, self.x_n = (tf.keras.layers.Dense(units) for _ in range(3))
        self.h_z, self.h_r, self.h_n = (tf.keras.layers.Dense(units, use_bias=False) for _ in range(3))
        self.activation_quantizers = {name: ModelOwnedQuantizer(bits=8, symmetric=False) for name in ("x_z", "x_r", "x_n", "h_z", "h_r", "h_n")}
    def call(self, x, h):
        x_z, x_r, x_n = (qat_dense(layer, x, self.activation_quantizers[name]) for name, layer in zip(("x_z", "x_r", "x_n"), (self.x_z, self.x_r, self.x_n)))
        h_z, h_r, h_n = (qat_dense(layer, h, self.activation_quantizers[name]) for name, layer in zip(("h_z", "h_r", "h_n"), (self.h_z, self.h_r, self.h_n)))
        z, r = tf.sigmoid(x_z + h_z), tf.sigmoid(x_r + h_r)
        h = z * h + (1.0 - z) * tf.tanh(x_n + r * h_n)
        return h, h

class PrimitiveLSTM(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__(); self.x_proj = tf.keras.layers.Dense(4 * units); self.h_proj = tf.keras.layers.Dense(4 * units, use_bias=False)
    def call(self, x, h, c):
        i, f, g, o = tf.split(qat_dense(self.x_proj, x) + qat_dense(self.h_proj, h), 4, axis=-1)
        c = tf.sigmoid(f) * c + tf.sigmoid(i) * tf.tanh(g)
        h = tf.sigmoid(o) * tf.tanh(c)
        return h, h, c
    def temporal_lstm_binned(self, x, h, c):
        """Independent temporal bins share one LSTM; reshape only changes GPU batching."""
        batch, bins = tf.shape(x)[0], tf.shape(x)[1]
        # QAT temporal observer update is per logical [B,F,H] boundary.
        x = tf.reshape(x, [batch * bins, HIDDEN_SIZE])
        h = tf.reshape(h, [batch * bins, HIDDEN_SIZE])
        c = tf.reshape(c, [batch * bins, HIDDEN_SIZE])
        value, h, c = self(x, h, c)
        shape = [batch, bins, HIDDEN_SIZE]
        return tf.reshape(value, shape), tf.reshape(h, shape), tf.reshape(c, shape)

QAT_ENABLED = False
class ModelOwnedQuantizer(tf.keras.layers.Layer):
    """Per-tensor asymmetric moving observer with an LSQ trainable step."""
    def __init__(self, bits=8, symmetric=False): super().__init__(); self.bits, self.symmetric = bits, symmetric
    def build(self, _):
        self.lsq_step = self.add_weight(name="lsq_step", shape=(), initializer=tf.keras.initializers.Constant(1e-2), trainable=True)
        self.moving_min = self.add_weight(name="moving_min", shape=(), initializer="zeros", trainable=False)
        self.moving_max = self.add_weight(name="moving_max", shape=(), initializer="ones", trainable=False)
    def call(self, x):
        if QAT_ENABLED:
            self.moving_min.assign(0.99 * self.moving_min + 0.01 * tf.reduce_min(tf.stop_gradient(x)))
            self.moving_max.assign(0.99 * self.moving_max + 0.01 * tf.reduce_max(tf.stop_gradient(x)))
        lo, hi = (-128.0, 127.0) if self.symmetric else (0.0, 255.0)
        step = tf.maximum(tf.abs(self.lsq_step), tf.constant(1e-8, x.dtype)); zero_point = tf.clip_by_value(tf.round(-self.moving_min / step), lo, hi)
        rounded = (tf.round(tf.clip_by_value(x / step + zero_point, lo, hi)) - zero_point) * step
        return x + tf.stop_gradient(rounded - x)

class CausalConv2D(tf.keras.layers.Layer):
    """Time-causal and frequency-same convolution over STFT frame history."""
    def __init__(self, filters, kernel_size=(3, 3), activation=None, weight_quantized=True):
        super().__init__(); self.kernel_size = kernel_size; self.weight_quantized = weight_quantized; self.layer = tf.keras.layers.Conv2D(filters, kernel_size, padding="valid", activation=activation)
    def call(self, inputs):
        kt, kf = self.kernel_size; left, right = (kf - 1) // 2, kf // 2
        return qat_conv2d(self.layer, tf.pad(inputs, [[0, 0], [kt - 1, 0], [left, right], [0, 0]]), self.weight_quantized)

QAT_ACTIVATION_QUANTIZER = ModelOwnedQuantizer(bits=8, symmetric=False)
def fake_quant_internal(x, quantizer=None):
    return (quantizer or QAT_ACTIVATION_QUANTIZER)(x) if QAT_ENABLED else x

def quantize_weight_per_channel(weight):
    """INT8 symmetric per-output-channel fake quant with an STE FP32 master."""
    if not QAT_ENABLED: return weight
    axes = list(range(weight.shape.rank - 1)); maximum = tf.reduce_max(tf.abs(weight), axis=axes, keepdims=True)
    step = tf.maximum(maximum / 127.0, tf.constant(1e-8, weight.dtype))
    projected = tf.round(tf.clip_by_value(weight / step, -127.0, 127.0)) * step
    return weight + tf.stop_gradient(projected - weight)

def qat_dense(layer, inputs, quantizer=None):
    baseline = layer(inputs)
    if not QAT_ENABLED: return baseline
    output = tf.tensordot(inputs, quantize_weight_per_channel(layer.kernel), axes=[[-1], [0]])
    if layer.use_bias: output = tf.nn.bias_add(output, layer.bias)
    return fake_quant_internal(output, quantizer)

def qat_conv2d(layer, inputs, weight_quantized=True, quantizer=None):
    baseline = layer(inputs)
    if not QAT_ENABLED or not weight_quantized: return baseline
    output = tf.nn.conv2d(inputs, quantize_weight_per_channel(layer.kernel), strides=[1, *layer.strides, 1], padding=layer.padding.upper(), dilations=[1, *layer.dilation_rate, 1])
    if layer.use_bias: output = tf.nn.bias_add(output, layer.bias)
    if layer.activation is not None: output = layer.activation(output)
    return fake_quant_internal(output, quantizer)

def qat_deconv2d(layer, inputs, weight_quantized=True, quantizer=None):
    baseline = layer(inputs)
    if not QAT_ENABLED or not weight_quantized: return baseline
    output = tf.nn.conv2d_transpose(inputs, quantize_weight_per_channel(layer.kernel), tf.shape(baseline), strides=[1, *layer.strides, 1], padding=layer.padding.upper())
    if layer.use_bias: output = tf.nn.bias_add(output, layer.bias)
    if layer.activation is not None: output = layer.activation(output)
    return fake_quant_internal(output, quantizer)

def state_shapes(batch=1):
    # NN state only. OLA/iSTFT cache is carried by initial_reconstruction_state.
    return {"conv_history": (batch, 2, FFT_BINS, 2), "dual_path_history": (batch, DECODER_TEMPORAL_KERNEL - 1, FFT_BINS, MODEL_DIM), **{f"lstm_h_{i}": (batch, COMPRESSED_BINS, HIDDEN_SIZE) for i in range(DUAL_PATH_BLOCKS)}, **{f"lstm_c_{i}": (batch, COMPRESSED_BINS, HIDDEN_SIZE) for i in range(DUAL_PATH_BLOCKS)}}

def initial_state(batch=1):
    state = {name: tf.zeros(shape, tf.float32) for name, shape in state_shapes(batch).items()}
    state["dual_path_history"] = tf.zeros([batch, DECODER_TEMPORAL_KERNEL - 1, FFT_BINS, MODEL_DIM], tf.float32)
    return state


## NeuralAid frame core

**Mục tiêu:** Thực hiện riêng nhiệm vụ: NeuralAid frame core.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách NeuralAid frame core; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
# Track B: U55-equivalent stride-2 decomposition; I/O and state contract are identical across tracks.
# PAPER-CONFIRMED: six dual-path blocks and causal decoder context.
# IMPLEMENTATION ASSUMPTION: kernels/padding are (3,3)/same because the paper
# does not publish a complete layer table.
TRACK = "u55_equivalent"
QAT_MODE = "u55_full_int8"
class NeuralAidFrameCore(tf.keras.Model):
    def __init__(self):
        super().__init__(); self.qat_activation_quantizer = QAT_ACTIVATION_QUANTIZER
        self.activation_quantizers = {name: ModelOwnedQuantizer(bits=8, symmetric=False) for name in ("encoder", "compressed", "spectral", "decoder")}
        self.enc = CausalConv2D(MODEL_DIM, (3, 3), activation="relu", weight_quantized=(QAT_MODE == "u55_full_int8")); self.comp1 = [tf.keras.layers.Conv2D(MODEL_DIM, (1, 2), strides=(1, 2), padding="same") for _ in range(DUAL_PATH_BLOCKS)]
        self.comp2 = [tf.keras.layers.Conv2D(MODEL_DIM, (1, 2), strides=(1, 2), padding="same") for _ in range(DUAL_PATH_BLOCKS)]
        self.expand1 = [tf.keras.layers.Conv2DTranspose(MODEL_DIM, (1, 2), strides=(1, 2), padding="same") for _ in range(DUAL_PATH_BLOCKS)]
        self.expand2 = [tf.keras.layers.Conv2DTranspose(MODEL_DIM, (1, 2), strides=(1, 2), padding="same") for _ in range(DUAL_PATH_BLOCKS)]
        # The unpublished layer table does not fix kernel size; k_t=3 is an
        # implementation assumption, with the two preceding chunks explicit in state.
        self.dec = CausalConv2D(2, (DECODER_TEMPORAL_KERNEL, 3), weight_quantized=True)
        self.spectral = [U55PrimitiveGRU(HIDDEN_SIZE) for _ in range(DUAL_PATH_BLOCKS)]
        self.temporal = [PrimitiveLSTM(HIDDEN_SIZE) for _ in range(DUAL_PATH_BLOCKS)]
        self.temporal_in = [tf.keras.layers.Dense(HIDDEN_SIZE) for _ in range(DUAL_PATH_BLOCKS)]
        self.temporal_out = [tf.keras.layers.Dense(MODEL_DIM) for _ in range(DUAL_PATH_BLOCKS)]
    def quantize_activation(self, name, value):
        if QAT_MODE == "paper_mixed_reference" and name in {"encoder", "decoder"}: return value
        return fake_quant_internal(value, self.activation_quantizers[name])
    def call(self, item):
        current = tf.expand_dims(item["tf_frame"], 1)
        x = self.quantize_activation("encoder", self.enc(tf.concat([item["conv_history"], current], 1))[:, -1:, :, :])
        out = {"next_conv_history": tf.concat([item["conv_history"][:, 1:], current], 1)}
        for block in range(DUAL_PATH_BLOCKS):
            x = qat_conv2d(self.comp2[block], qat_conv2d(self.comp1[block], x)); compressed = self.quantize_activation("compressed", tf.squeeze(x, 1)); spectral = []; h = tf.zeros([tf.shape(compressed)[0], HIDDEN_SIZE])
            for freq in range(COMPRESSED_BINS): h, value = self.spectral[block](compressed[:, freq, :], h); spectral.append(value)
            spectral = tf.stack(spectral, 1)
            temporal_input = qat_dense(self.temporal_in[block], spectral)
            temporal_value, temporal_h, temporal_c = self.temporal[block].temporal_lstm_binned(temporal_input, item[f"lstm_h_{block}"], item[f"lstm_c_{block}"])
            spectral = self.quantize_activation("spectral", spectral + qat_dense(self.temporal_out[block], temporal_value)); x = tf.expand_dims(spectral, 1); x = qat_deconv2d(self.expand2[block], qat_deconv2d(self.expand1[block], x)); x = fake_quant_internal(x[:, :, :FFT_BINS, :])
            out[f"next_lstm_h_{block}"] = temporal_h; out[f"next_lstm_c_{block}"] = temporal_c
        current_dual = x
        decoder_context = tf.concat([item["dual_path_history"], current_dual], 1)
        decoded_feature = self.quantize_activation("decoder", self.dec(decoder_context)[:, -1:, :FFT_BINS, :])
        out["next_dual_path_history"] = tf.concat([item["dual_path_history"][:, 1:], current_dual], 1)
        out["enhanced_tf"] = tf.squeeze(decoded_feature, 1)
        return out
with tf.device(GPU_TRAIN_DEVICE):
    FRAME_CORE = NeuralAidFrameCore(); _ = FRAME_CORE({"tf_frame": tf.zeros([1, FFT_BINS, 2]), **initial_state(1)})


## Preflight INT8 export

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Preflight INT8 export.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Preflight INT8 export; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
PACKAGE = RUN_DIR / "integration_package"; PACKAGE.mkdir(parents=True, exist_ok=True)
def representative_frames(max_frames=128):
    samples = []
    for rel in val_pairs.iloc[:8]["mixture_path"]:
        audio = read_audio(DATASET_ROOT / rel)[None, :]; state = initial_state(1)
        contexts = make_stream_contexts(audio)
        for index in range(int(tf.shape(contexts)[1].numpy())):
            context = contexts[:, index, :]
            spectrum = frontend_frame(context); frame = tf.stack([tf.math.real(spectrum), tf.math.imag(spectrum)], -1)
            samples.append({"tf_frame": frame.numpy().astype(np.float32), **{k:v.numpy().astype(np.float32) for k,v in state.items()}})
            result = FRAME_CORE({"tf_frame": frame, **state}); state = {k: result[f"next_{k}"] for k in state}
            if len(samples) >= max_frames: return samples
    if not samples: raise RuntimeError("no calibration frames")
    return samples
def export_int8(path, samples, core=None):
    core = core or FRAME_CORE; state = initial_state(1)
    signature = {"tf_frame": tf.TensorSpec([1, FFT_BINS, 2], tf.float32, "tf_frame"), **{k: tf.TensorSpec(v.shape, tf.float32, k) for k,v in state.items()}}
    class Exporter(tf.Module):
        def __init__(self, model): super().__init__(); self.model = model
        @tf.function
        def serve(self, inputs): return self.model(inputs)
    exporter = Exporter(core); concrete = exporter.serve.get_concrete_function(signature)
    # Direct TFLite conversion can replace dict output names with
    # StatefulPartitionedCall:*; its flattened output order is preserved.
    global EXPORT_LOGICAL_OUTPUT_ORDER
    # tf.nest flattens dictionary outputs in sorted-key order for conversion.
    EXPORT_LOGICAL_OUTPUT_ORDER = sorted(concrete.structured_outputs)
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete], exporter); converter.optimizations = [tf.lite.Optimize.DEFAULT]
    names = [t.name.split(":")[0] for t in concrete.inputs if t.dtype != tf.resource]
    converter.representative_dataset = lambda: ([np.asarray(item[name], np.float32) for name in names] for item in samples)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]; converter.inference_input_type = tf.int8; converter.inference_output_type = tf.int8
    Path(path).write_bytes(converter.convert()); return Path(path)
PREFLIGHT_TFLITE = export_int8(PACKAGE / "neuralaid_preflight_int8.tflite", representative_frames())


## Vela audit

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Vela audit.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Vela audit; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
VELA_ACCELERATOR = "ethos-u55-256"
VELA_CONFIG = """[System_Config.M55M1_U55_256]
core_clock=220e6
axi0_port=Sram
axi1_port=OffChipFlash
Sram_clock_scale=1.0
Sram_burst_length=32
Sram_read_latency=32
Sram_write_latency=32
Sram_max_reads=4
Sram_max_writes=4
OffChipFlash_clock_scale=0.125
OffChipFlash_burst_length=128
OffChipFlash_read_latency=64
OffChipFlash_write_latency=0
OffChipFlash_max_reads=2
OffChipFlash_max_writes=0
[Memory_Mode.M55M1_Shared_SRAM]
const_mem_area=Axi1
arena_mem_area=Axi0
cache_mem_area=Axi0
arena_cache_size=1572864
"""
VELA_CONFIG_PATH = PACKAGE / "vela_m55m1.ini"; VELA_CONFIG_PATH.write_text(VELA_CONFIG, encoding="utf-8")
def vela_audit(model, prefix):
    variants = []
    for mode in ("Size", "Performance"):
        folder = PACKAGE / f"vela_{prefix}_{mode.lower()}"; folder.mkdir(exist_ok=True)
        command = [VELA_BIN, str(model), "--accelerator-config", VELA_ACCELERATOR, "--config", str(VELA_CONFIG_PATH), "--system-config", "M55M1_U55_256", "--memory-mode", "M55M1_Shared_SRAM", "--optimise", mode, "--output-dir", str(folder), "--show-cpu-operations"]
        run = subprocess.run(command, text=True, capture_output=True); report = f"COMMAND: {' '.join(command)}\nEXIT_STATUS: {run.returncode}\nSTDOUT:\n{run.stdout}\nSTDERR:\n{run.stderr}\n"
        report_name = f"vela_report_{prefix}_{mode.lower()}.txt"; (PACKAGE / report_name).write_text(report, encoding="utf-8"); models = list(folder.rglob("*_vela.tflite"))
        variants.append({"mode":mode,"exit_status":run.returncode,"cpu_operators_zero":run.returncode == 0 and "CPU operators = 0" in report,"vela_tflite":str(models[0]) if len(models)==1 else None,"report":report_name})
    ready = any(v["cpu_operators_zero"] for v in variants); audit = {"track":TRACK,"input_tflite":str(model),"accelerator":VELA_ACCELERATOR,"variants":variants,"deploy_ready":ready,"deployment_status":"deploy_ready" if ready else "research_only"}
    (PACKAGE / f"{prefix}_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8"); (PACKAGE / "preflight_audit.json" if prefix == "preflight" else PACKAGE / "final_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8"); return audit
PREFLIGHT_AUDIT = vela_audit(PREFLIGHT_TFLITE, "preflight")


## FP32 train and validation

**Mục tiêu:** Thực hiện riêng nhiệm vụ: FP32 train and validation.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách FP32 train and validation; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
# A 5-second crop is paper-base only.  This board-response dataset
# declares four-second clips, so project adaptation crops real 4-second audio
# rather than zero-padding it and mislabelling the result as paper reproduction.
FP32_WALL_CLOCK_SECONDS = RUN_CONFIG["fp32_deadline_seconds"]
fp32_resume_path = RUN_DIR / "fp32_resume_state.json"
PAPER_TRAIN_CROP_SECONDS = 5
TRAINING_MODE = "paper_base" if DATASET_PROVENANCE.get("paper_base_recipe", False) else "project_adaptation"
TRAIN_CROP_SECONDS = PAPER_TRAIN_CROP_SECONDS if TRAINING_MODE == "paper_base" else float(DATASET_PROVENANCE["clip_seconds"])
FULL_TRAIN_CROP_SAMPLES = int(SAMPLE_RATE * TRAIN_CROP_SECONDS)
DEBUG_FULL_STREAM_CHUNKS = math.ceil(FULL_TRAIN_CROP_SAMPLES / CHUNK_SAMPLES)
TRAIN_CROP_SAMPLES = int(RUN_CONFIG.get("chunks", 0) * CHUNK_SAMPLES) if RUN_PROFILE == "debug" else FULL_TRAIN_CROP_SAMPLES
if TRAINING_MODE == "project_adaptation" and RUN_PROFILE != "debug": assert TRAIN_CROP_SAMPLES == int(DATASET_PROVENANCE["samples_per_clip"]), "custom data must not be padded to the paper 5-s crop"
if RUN_PROFILE == "paper_base" and TRAINING_MODE != "paper_base": raise ValueError("paper_base requires >=5 s paper-recipe clips")
PAPER_STEPS_PER_EPOCH = int(RUN_CONFIG["paper_steps_per_epoch"])
if RUN_PROFILE == "p100_longrun":
    assert PAPER_STEPS_PER_EPOCH * int(RUN_CONFIG["batch"]) == PAPER_SAMPLES_PER_EPOCH, "paper schedule requires exactly 20,000 mixtures/epoch"
TRAIN_CHUNKS = math.ceil(TRAIN_CROP_SAMPLES / CHUNK_SAMPLES)
GPU_RUNTIME = require_gpu_for_training()

def write_run_manifest():
    manifest = {"track": TRACK, "run_profile": RUN_PROFILE, "training_mode": TRAINING_MODE,
                "sample_rate": SAMPLE_RATE, "train_crop_seconds": TRAIN_CROP_SECONDS,
                "experiment_id": EXPERIMENT_ID, "paper_steps_per_epoch": PAPER_STEPS_PER_EPOCH, "batch_size": int(RUN_CONFIG["batch"]),
                "samples_per_paper_epoch": PAPER_STEPS_PER_EPOCH * int(RUN_CONFIG["batch"]), "train_chunks": TRAIN_CHUNKS,
                "fp32_session_epochs": RUN_CONFIG["fp32_session_epochs"], "fp32_steps_per_session_epoch": RUN_CONFIG["fp32_steps_per_session_epoch"],
                "qat_session_epochs": RUN_CONFIG["qat_session_epochs"], "qat_steps_per_session_epoch": RUN_CONFIG["qat_steps_per_session_epoch"],
                "lr_schedule_kind": RUN_CONFIG["lr_schedule_kind"], "warmup_steps": RUN_CONFIG["warmup_steps"],
                "temporal_parallelism": "disabled", "batch_parallelism": "independent_utterances_only",
                "temporal_lstm_vectorized": True, "gpu_runtime": GPU_RUNTIME,
                "training_budget": RUN_CONFIG, "dataset_provenance": DATASET_PROVENANCE}
    path = RUN_DIR / "run_manifest.json"; path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest
RUN_MANIFEST = write_run_manifest()
def initial_model_state(batch_size): return {k: tf.zeros(v, tf.float32) for k, v in state_shapes(batch_size).items()}
def enhance_sequence(waveform):
    # The compiled train step supplies a fixed 24-chunk crop in debug; evaluation
    # and endurance retain the same causal loop but derive their stream length.
    contexts = make_stream_contexts(waveform); chunk_count = tf.shape(contexts)[1]
    state = initial_model_state(tf.shape(waveform)[0]); reconstruction = initial_reconstruction_state(tf.shape(waveform)[0]); chunks = tf.TensorArray(tf.float32, size=chunk_count)
    def body(index, state, reconstruction, chunks):
        context = contexts[:, index, :]
        spectrum = frontend_frame(context); frame = tf.stack([tf.math.real(spectrum), tf.math.imag(spectrum)], -1)
        result = FRAME_CORE({"tf_frame": frame, **state}); state = {k: result[f"next_{k}"] for k in state}
        short = normalized_reference_synthesis(tf.complex(result["enhanced_tf"][..., 0], result["enhanced_tf"][..., 1]))
        emitted, reconstruction["ola_tail"], reconstruction["ola_weight_tail"] = synthesis_stream(short, reconstruction["ola_tail"], reconstruction["ola_weight_tail"])
        return index + 1, state, reconstruction, chunks.write(index, emitted)
    _, _, _, chunks = tf.while_loop(lambda index, *_: index < chunk_count, body, (tf.constant(0), state, reconstruction, chunks), parallel_iterations=1)
    output = tf.transpose(chunks.stack(), [1, 0, 2]); output = tf.reshape(output, [tf.shape(waveform)[0], -1])
    return output[:, :tf.shape(waveform)[1]]
def debug_stream_endurance():
    """Inference-only full clip check; temporal chunks remain causal and serial."""
    if RUN_PROFILE != "debug": return None
    waveform = tf.zeros([1, DEBUG_FULL_STREAM_CHUNKS * CHUNK_SAMPLES], tf.float32)
    contexts, state, reconstruction, emitted = make_stream_contexts(waveform), initial_model_state(1), initial_reconstruction_state(1), []
    for index in range(int(tf.shape(contexts)[1].numpy())):
        spectrum = frontend_frame(contexts[:, index, :]); frame = tf.stack([tf.math.real(spectrum), tf.math.imag(spectrum)], -1)
        result = FRAME_CORE({"tf_frame": frame, **state}); state = {key: result[f"next_{key}"] for key in state}
        short = normalized_reference_synthesis(tf.complex(result["enhanced_tf"][..., 0], result["enhanced_tf"][..., 1]))
        output, reconstruction["ola_tail"], reconstruction["ola_weight_tail"] = synthesis_stream(short, reconstruction["ola_tail"], reconstruction["ola_weight_tail"]); emitted.append(output)
    output = tf.concat(emitted, 1); tf.debugging.assert_all_finite(output, "debug endurance output"); tf.debugging.assert_equal(tf.shape(output)[1], DEBUG_FULL_STREAM_CHUNKS * CHUNK_SAMPLES)
    return {"chunks": DEBUG_FULL_STREAM_CHUNKS, "samples": int(tf.shape(output)[1])}
def snr_loss(reference, estimate):
    n = tf.minimum(tf.shape(reference)[1], tf.shape(estimate)[1]); reference, estimate = reference[:, :n], estimate[:, :n]; noise = reference - estimate
    return -tf.reduce_mean(10.0 * tf.math.log((tf.reduce_sum(reference ** 2, 1) + 1e-8) / (tf.reduce_sum(noise ** 2, 1) + 1e-8)) / tf.math.log(10.0))
class PaperBaseSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        if RUN_CONFIG["lr_schedule_kind"] == "budget_normalized":
            warmup_steps = float(max(RUN_CONFIG["warmup_steps"], 1))
            return tf.where(step < warmup_steps, 1e-4 + tf.minimum(step / warmup_steps, 1.0) * 9e-4, 1e-3)
        epoch = step / float(max(PAPER_STEPS_PER_EPOCH, 1)); warm = tf.minimum(epoch / 10.0, 1.0); base = tf.where(epoch <= 10.0, 1e-4 + warm * 9e-4, 1e-3); return tf.where(epoch > 150.0, base * tf.pow(0.5, tf.maximum(tf.floor((epoch - 150.0) / 15.0), 0.0)), base)
optimizer = tf.keras.optimizers.AdamW(PaperBaseSchedule(), clipnorm=0.1)
def make_training_dataset(frame, batch_size, training):
    mixtures = [str(DATASET_ROOT / p) for p in frame["mixture_path"]]; cleans = [str(DATASET_ROOT / p) for p in frame["clean_path"]]
    dataset = tf.data.Dataset.from_tensor_slices((mixtures, cleans))
    if training: dataset = dataset.shuffle(min(len(frame), 4096), seed=SEED, reshuffle_each_iteration=True).repeat()
    def load_pair(mixture_path, clean_path):
        mixture, clean = read_audio(mixture_path), read_audio(clean_path); length = tf.minimum(tf.shape(mixture)[0], tf.shape(clean)[0]); mixture, clean = mixture[:length], clean[:length]
        tf.debugging.assert_greater_equal(length, TRAIN_CROP_SAMPLES, message="clip shorter than declared training crop")
        if training:
            start = tf.random.uniform([], 0, length - TRAIN_CROP_SAMPLES + 1, dtype=tf.int32); mixture, clean = mixture[start:start+TRAIN_CROP_SAMPLES], clean[start:start+TRAIN_CROP_SAMPLES]
        else: mixture, clean = mixture[:TRAIN_CROP_SAMPLES], clean[:TRAIN_CROP_SAMPLES]
        return mixture, clean
    with tf.device("/CPU:0"):
        dataset = dataset.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size, drop_remainder=training)
    return dataset.prefetch(tf.data.AUTOTUNE)
TRAIN_INPUT_SIGNATURE = (tf.TensorSpec([None, TRAIN_CROP_SAMPLES], tf.float32), tf.TensorSpec([None, TRAIN_CROP_SAMPLES], tf.float32))
@tf.function(input_signature=(TRAIN_INPUT_SIGNATURE[0],), reduce_retracing=True)
def fp32_enhance_sequence(waveform):
    with tf.device(GPU_TRAIN_DEVICE): return enhance_sequence(waveform)
# Training keeps a fixed 4-s signature for stable GPU traces.  Test clips are
# full-length 4-s files today but must not be coupled to that crop contract.
@tf.function(input_signature=(tf.TensorSpec([None, None], tf.float32),), reduce_retracing=True)
def evaluation_enhance_sequence(waveform):
    with tf.device(GPU_TRAIN_DEVICE): return enhance_sequence(waveform)
@tf.function(input_signature=TRAIN_INPUT_SIGNATURE, reduce_retracing=True)
def train_step(mixture, clean):
    with tf.device(GPU_TRAIN_DEVICE):
        with tf.GradientTape() as tape: loss = snr_loss(clean, fp32_enhance_sequence(mixture))
        gradients = tape.gradient(loss, FRAME_CORE.trainable_variables); optimizer.apply_gradients(zip(gradients, FRAME_CORE.trainable_variables)); return loss
def validation_loss(max_batches=1 if RUN_PROFILE == "debug" else 8):
    values = [float(snr_loss(clean, fp32_enhance_sequence(mixture))) for mixture, clean in make_training_dataset(val_pairs, RUN_CONFIG["batch"], False).take(max_batches)]
    return float(np.mean(values)) if values else float("inf")

# Paper reports SI-SDRi, PESQ and personalized DNSMOS P.835.  Full 3,750-item
# evaluation is intentionally opt-in because this streaming reference graph is
# expensive; every report records its evaluated test-shard size.
EVALUATION_MAX_ITEMS = RUN_CONFIG["evaluation_max_items"]
def si_sdr_db(reference, estimate):
    reference, estimate = np.asarray(reference, np.float64), np.asarray(estimate, np.float64)
    reference = reference - reference.mean(); estimate = estimate - estimate.mean()
    target = np.dot(estimate, reference) * reference / (np.dot(reference, reference) + 1e-12)
    return float(10.0 * np.log10((np.dot(target, target) + 1e-12) / (np.dot(estimate - target, estimate - target) + 1e-12)))

def pesq_wb(reference, estimate):
    try:
        from pesq import pesq
        return float(pesq(SAMPLE_RATE, reference, estimate, "wb")), None
    except Exception as error:
        return None, f"{type(error).__name__}: {error}"

def stoi_score(reference, estimate):
    try:
        from pystoi import stoi
        return float(stoi(reference, estimate, SAMPLE_RATE, extended=False)), None
    except Exception as error:
        return None, f"{type(error).__name__}: {error}"

def dnsmos_p835():
    # The official personalized P.835 evaluator requires its two ONNX assets.
    # Keep the paper metric explicit rather than substituting an unrelated MOS.
    return {"status": "not_run", "reason": "supply official DNSMOS/pDNSMOS ONNX assets to enable P.835 evaluation"}

def evaluate_speech_enhancement(tag, max_items=EVALUATION_MAX_ITEMS, enhancement_fn=evaluation_enhance_sequence):
    if max_items is None:
        pairs, evaluation_kind = test_pairs, "full_test"
    else:
        indices = np.sort(np.random.default_rng(SEED).choice(len(test_pairs), int(max_items), replace=False))
        pairs, evaluation_kind = test_pairs.iloc[indices], "representative_quick"
    rows = []
    for item in pairs.itertuples(index=False):
        mixture = read_audio(DATASET_ROOT / item.mixture_path)[None, :]
        clean = read_audio(DATASET_ROOT / item.clean_path)[None, :]
        enhanced = enhancement_fn(mixture)
        length = int(tf.minimum(tf.shape(mixture)[1], tf.minimum(tf.shape(clean)[1], tf.shape(enhanced)[1])).numpy())
        mix, reference, output = mixture.numpy()[0, :length], clean.numpy()[0, :length], enhanced.numpy()[0, :length]
        pesq_score, pesq_error = pesq_wb(reference, output)
        stoi_value, stoi_error = stoi_score(reference, output)
        input_si_sdr, output_si_sdr = si_sdr_db(reference, mix), si_sdr_db(reference, output)
        rows.append({"mixture_id": str(item.mixture_id), "si_sdr_input_db": input_si_sdr, "si_sdr_output_db": output_si_sdr, "si_sdri": output_si_sdr - input_si_sdr, "pesq_wb": pesq_score, "pesq_error": pesq_error, "stoi": stoi_value, "stoi_error": stoi_error})
    if not rows: raise RuntimeError("test evaluation received no pairs")
    si_input = np.asarray([row["si_sdr_input_db"] for row in rows], np.float64); si_output = np.asarray([row["si_sdr_output_db"] for row in rows], np.float64)
    pesq_scores = [row["pesq_wb"] for row in rows if row["pesq_wb"] is not None]
    stoi_scores = [row["stoi"] for row in rows if row["stoi"] is not None]
    report = {"track": TRACK, "tag": tag, "split": "test", "evaluation_kind": evaluation_kind, "evaluation_sampling_seed": SEED if max_items is not None else None, "evaluated_items": len(rows), "test_items_total": len(test_pairs), "evaluation_max_items": max_items,
              "si_sdr_input_db": float(np.mean(si_input)), "si_sdr_output_db": float(np.mean(si_output)), "si_sdri": float(np.mean(si_output - si_input)),
              "pesq_wb": {"mean": float(np.mean(pesq_scores)) if pesq_scores else None, "scored_items": len(pesq_scores), "failed_items": len(rows) - len(pesq_scores)},
              "stoi": {"mean": float(np.mean(stoi_scores)) if stoi_scores else None, "scored_items": len(stoi_scores), "failed_items": len(rows) - len(stoi_scores)},
              "dnsmos_p835": dnsmos_p835(), "samples": rows}
    path = RUN_DIR / f"metrics_{tag}.json"; path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print({"post_train_metrics": tag, "evaluation_kind": evaluation_kind, "evaluated_items": report["evaluated_items"], "si_sdri": report["si_sdri"], "pesq_wb": report["pesq_wb"]["mean"], "stoi": report["stoi"]["mean"], "dnsmos_p835": report["dnsmos_p835"]["status"]})
    return report

def run_preflight_tests():
    """Cheap deterministic gates that execute before a long FP32 run."""
    assert np.allclose(SYNTHESIS_WINDOW[:LOOKAHEAD_SAMPLES], SYNTHESIS_EDGE_GAIN)
    assert np.allclose(SYNTHESIS_WINDOW[LOOKAHEAD_SAMPLES:CHUNK_SAMPLES], 1.0)
    assert np.allclose(SYNTHESIS_WINDOW[CHUNK_SAMPLES:], SYNTHESIS_EDGE_GAIN)
    probe_length = 1536
    time = tf.range(probe_length, dtype=tf.float32)[None, :] / float(SAMPLE_RATE)
    speech_clean = read_audio(str(DATASET_ROOT / val_pairs.iloc[0]["clean_path"]))[None, :probe_length]
    probes = {
        "impulse": tf.concat([tf.ones([1, 1]), tf.zeros([1, probe_length - 1])], 1),
        "tone_440": tf.sin(2.0 * np.pi * 440.0 * time),
        "tone_3k": tf.sin(2.0 * np.pi * 3000.0 * time),
        "noise": tf.random.stateless_normal([1, probe_length], seed=[SEED, 17]),
        "speech_clean": speech_clean,
    }
    errors = {}
    for name, probe in probes.items():
        length = int(tf.shape(probe)[1].numpy())
        expected = pre_emphasis_sequence(probe); reconstruction = initial_reconstruction_state(1); chunks = []; contexts = make_stream_contexts(probe)
        for index in range(int(tf.shape(contexts)[1].numpy())):
            short = normalized_reference_synthesis(frontend_frame(contexts[:, index, :]))
            emitted, reconstruction["ola_tail"], reconstruction["ola_weight_tail"] = synthesis_stream(short, reconstruction["ola_tail"], reconstruction["ola_weight_tail"]); chunks.append(emitted)
        errors[name] = float(tf.reduce_max(tf.abs(expected - tf.concat(chunks, 1)[:, :length])))
    layer = FRAME_CORE.temporal[0]; x = tf.random.stateless_normal([1, COMPRESSED_BINS, HIDDEN_SIZE], [SEED, 101]); h = tf.random.stateless_normal([1, COMPRESSED_BINS, HIDDEN_SIZE], [SEED, 102]); c = tf.random.stateless_normal([1, COMPRESSED_BINS, HIDDEN_SIZE], [SEED, 103])
    reference = [layer(x[:, freq, :], h[:, freq, :], c[:, freq, :]) for freq in range(COMPRESSED_BINS)]
    vectorized = layer.temporal_lstm_binned(x, h, c)
    temporal_parity = max(float(tf.reduce_max(tf.abs(tf.stack([part[0] for part in reference], 1) - vectorized[0]))), float(tf.reduce_max(tf.abs(tf.stack([part[1] for part in reference], 1) - vectorized[1]))), float(tf.reduce_max(tf.abs(tf.stack([part[2] for part in reference], 1) - vectorized[2]))))
    if temporal_parity > 1e-6: raise AssertionError({"temporal_lstm_vectorization_error": temporal_parity})
    state = initial_state(1); frame = tf.zeros([1, FFT_BINS, 2], tf.float32)
    baseline = FRAME_CORE({"tf_frame": frame, **state})
    changed = dict(state); changed["dual_path_history"] = tf.ones_like(changed["dual_path_history"])
    history_out = FRAME_CORE({"tf_frame": frame, **changed})
    history_delta = float(tf.reduce_max(tf.abs(baseline["enhanced_tf"] - history_out["enhanced_tf"])))
    if max(errors.values()) >= 1e-5 or history_delta <= 0.0: raise AssertionError({"dsp_errors": errors, "decoder_history_delta": history_delta})
    return {"dsp_identity_max_error": max(errors.values()), "decoder_history_delta": history_delta, "temporal_lstm_vectorization_error": temporal_parity, "state_names": sorted(state)}

PREFLIGHT_RESULT = run_preflight_tests()
PREFLIGHT_RESULT["debug_stream_endurance"] = debug_stream_endurance()
(RUN_DIR / "preflight.json").write_text(json.dumps(PREFLIGHT_RESULT, indent=2), encoding="utf-8")
if RUN_PROFILE == "p100_full_eval": FRAME_CORE.load_weights(FULL_EVAL_CHECKPOINT, skip_mismatch=True)
PROGRESS_INTERVAL_STEPS = 25
VALIDATION_INTERVAL_STEPS = 250
FP32_CHECKPOINT = tf.train.Checkpoint(model=FRAME_CORE, optimizer=optimizer)
FP32_MANAGER = tf.train.CheckpointManager(FP32_CHECKPOINT, str(RUN_DIR / "checkpoints" / "fp32"), max_to_keep=2)
def append_progress(path, payload):
    with path.open("a", encoding="utf-8") as stream: stream.write(json.dumps(payload) + "\\n")
def budget_status(completed, target, deadline):
    return "completed" if completed >= target else ("deadline_limited" if time.monotonic() - SESSION_START >= deadline else "step_limited")
def session_position(completed_steps, steps_per_session_epoch):
    """Return the completed session epoch and its one-based local step."""
    if not steps_per_session_epoch or completed_steps <= 0: return 0, 0
    return ((completed_steps - 1) // steps_per_session_epoch + 1, (completed_steps - 1) % steps_per_session_epoch + 1)
def save_fp32_state(state, loss):
    state["checkpoint"] = FP32_MANAGER.save(checkpoint_number=state["completed_fp32_steps"])
    state["last_loss"] = float(loss); state["elapsed_seconds"] = time.monotonic() - SESSION_START
    temporary = fp32_resume_path.with_suffix(".tmp"); temporary.write_text(json.dumps(state, indent=2), encoding="utf-8"); temporary.replace(fp32_resume_path)
    return state
if RUN_PROFILE == "p100_full_eval":
    FRAME_CORE.load_weights(FULL_EVAL_CHECKPOINT, skip_mismatch=True)
    fp32_state = {"phase":"full_eval", "completed_fp32_steps":0, "training_budget_status":"not_run"}
    FP32_METRICS = None
else:
    fp32_state = json.loads(fp32_resume_path.read_text()) if fp32_resume_path.is_file() else {"phase":"fp32", "completed_fp32_steps":0, "best_validation_loss":float("inf")}
    if fp32_state.get("checkpoint"): FP32_CHECKPOINT.restore(fp32_state["checkpoint"]).expect_partial()
    fp32_start = int(fp32_state["completed_fp32_steps"]); fp32_loss = float(fp32_state.get("last_loss", 0.0)); fp32_clock = time.monotonic()
    train_iterator = iter(make_training_dataset(train_pairs, RUN_CONFIG["batch"], True))
    while fp32_state["completed_fp32_steps"] < RUN_CONFIG["fp32_max_steps"] and time.monotonic() - SESSION_START < RUN_CONFIG["fp32_deadline_seconds"]:
        completed = fp32_state["completed_fp32_steps"]; session_epoch = completed // RUN_CONFIG["fp32_steps_per_session_epoch"] + 1
        session_end = min(session_epoch * RUN_CONFIG["fp32_steps_per_session_epoch"], RUN_CONFIG["fp32_max_steps"])
        while fp32_state["completed_fp32_steps"] < session_end and time.monotonic() - SESSION_START < RUN_CONFIG["fp32_deadline_seconds"]:
            mixture, clean = next(train_iterator); fp32_loss = float(train_step(mixture, clean)); fp32_state["completed_fp32_steps"] += 1
            if fp32_state["completed_fp32_steps"] % PROGRESS_INTERVAL_STEPS == 0: fp32_state = save_fp32_state(fp32_state, fp32_loss)
        completed = fp32_state["completed_fp32_steps"]; epoch, local_step = session_position(completed, RUN_CONFIG["fp32_steps_per_session_epoch"])
        fp32_state = save_fp32_state(fp32_state, fp32_loss)
        current = validation_loss()
        if current < fp32_state["best_validation_loss"] or not (RUN_DIR / "best_fp32.weights.h5").is_file(): fp32_state["best_validation_loss"] = current; FRAME_CORE.save_weights(RUN_DIR / "best_fp32.weights.h5")
        elapsed = time.monotonic() - fp32_clock; append_progress(RUN_DIR / "progress_fp32.jsonl", {"phase":"fp32", "global_step":completed, "session_epoch":epoch, "step_in_session_epoch":local_step, "loss":fp32_loss, "learning_rate":float(PaperBaseSchedule()(optimizer.iterations).numpy()), "seconds_per_step":elapsed / max(completed - fp32_start, 1), "remaining_seconds":max(0.0, RUN_CONFIG["fp32_deadline_seconds"] - (time.monotonic() - SESSION_START)), "target_steps":RUN_CONFIG["fp32_max_steps"]})
    fp32_state["training_budget_status"] = budget_status(fp32_state["completed_fp32_steps"], RUN_CONFIG["fp32_max_steps"], RUN_CONFIG["fp32_deadline_seconds"])
    fp32_resume_path.write_text(json.dumps(fp32_state, indent=2), encoding="utf-8")
    best_fp32 = RUN_DIR / "best_fp32.weights.h5"
    if best_fp32.is_file(): FRAME_CORE.load_weights(best_fp32)
    FP32_METRICS = evaluate_speech_enhancement("fp32")
RUN_FINALIZATION = RUN_PROFILE == "p100_full_eval" or bool(RUN_CONFIG["final_in_session"]) or fp32_state["training_budget_status"] == "completed"


## QAT and final export

**Mục tiêu:** Thực hiện riêng nhiệm vụ: QAT and final export.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách QAT and final export; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
import time
def fake_quant(x):
    lo = tf.reduce_min(tf.stop_gradient(x)); hi = tf.reduce_max(tf.stop_gradient(x)); hi = tf.maximum(hi, lo + 1e-5)
    return tf.quantization.fake_quant_with_min_max_vars(x, lo, hi, num_bits=8, narrow_range=False)

def prepare_qat_model(model):
    """Audit the real fake-quant forward paths while retaining FP32 optimizer masters."""
    kernels = [v.name for v in model.trainable_variables if "kernel" in v.name]
    if not kernels: raise AssertionError("QAT model has no quantizable kernels")
    return {"qat_mode": QAT_MODE, "weight_policy": "per_channel_symmetric_kernel", "activation_policy": "per_site_asymmetric_lsq", "weight_quantization_coverage": kernels, "excluded": [v.name for v in model.trainable_variables if "bias" in v.name]}

class PlateauController:
    def __init__(self, optimizer, patience=5, factor=0.5): self.optimizer, self.patience, self.factor, self.best, self.bad_epochs = optimizer, patience, factor, float("inf"), 0
    def update(self, metric):
        if metric < self.best: self.best, self.bad_epochs = metric, 0
        else: self.bad_epochs += 1
        if self.bad_epochs >= self.patience: self.optimizer.learning_rate.assign(self.optimizer.learning_rate * self.factor); self.bad_epochs = 0

qat_optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, clipnorm=0.1)
qat_plateau = PlateauController(qat_optimizer, patience=5, factor=0.5)
QAT_SAMPLES_PER_EPOCH = 4000
QAT_STEPS_PER_EPOCH = 1 if RUN_PROFILE == "debug" else QAT_SAMPLES_PER_EPOCH // RUN_CONFIG["batch"]
QAT_WALL_CLOCK_SECONDS = RUN_CONFIG["qat_deadline_seconds"]
def benchmark_p100_train_step(steps=50):
    """Measure full-crop forward/backward cost without updating FP32 weights."""
    start = time.monotonic(); dataset = make_training_dataset(train_pairs, RUN_CONFIG["batch"], True)
    for mixture, clean in dataset.take(steps):
        with tf.GradientTape() as tape: loss = snr_loss(clean, enhance_sequence(mixture))
        _ = tape.gradient(loss, FRAME_CORE.trainable_variables)
    seconds_per_step = (time.monotonic() - start) / steps
    result = {"device_target": "Kaggle P100", "steps": steps, "seconds_per_step": seconds_per_step, "fp32_hours": 500000 * seconds_per_step / 3600, "qat_hours": 15000 * seconds_per_step / 3600}
    (RUN_DIR / "p100_benchmark.json").write_text(json.dumps(result, indent=2), encoding="utf-8"); return result
@tf.function(input_signature=TRAIN_INPUT_SIGNATURE, reduce_retracing=True)
def qat_train_step(mixture, clean):
    with tf.device(GPU_TRAIN_DEVICE):
        with tf.GradientTape() as tape:
            enhanced = enhance_sequence(mixture)
            loss = snr_loss(clean, enhanced)
        gradients = tape.gradient(loss, FRAME_CORE.trainable_variables); qat_optimizer.apply_gradients(zip(gradients, FRAME_CORE.trainable_variables)); return loss
def run_qat(max_steps):
    # QAT uses internal activation fake-quant and trainable-weight projection, not output-only quantization.
    global QAT_ENABLED; QAT_ENABLED = False
    fp32_checkpoint = RUN_DIR / "best_fp32.weights.h5"
    # FP32 checkpoints predate QAT-only observer variables.  Loading while the
    # observers are inactive retains all shared model weights and skips only
    # their absent LSQ/moving-range state, including on a rerun after QAT began.
    if fp32_checkpoint.is_file(): FRAME_CORE.load_weights(fp32_checkpoint, skip_mismatch=True)
    QAT_ENABLED = True
    qat_policy = prepare_qat_model(FRAME_CORE)
    resume_path = RUN_DIR / "qat_resume_state.json"; QAT_CHECKPOINT = tf.train.Checkpoint(model=FRAME_CORE, optimizer=qat_optimizer); QAT_MANAGER = tf.train.CheckpointManager(QAT_CHECKPOINT, str(RUN_DIR / "checkpoints" / "qat"), max_to_keep=2)
    resume = json.loads(resume_path.read_text()) if resume_path.is_file() else {"phase":"qat", "completed_qat_steps":0}
    if resume.get("checkpoint"): QAT_CHECKPOINT.restore(resume["checkpoint"]).expect_partial()
    qat_start = int(resume["completed_qat_steps"]); qat_loss = float(resume.get("last_loss", 0.0)); qat_clock = time.monotonic(); qat_iterator = iter(make_training_dataset(train_pairs, RUN_CONFIG["batch"], True))
    while resume["completed_qat_steps"] < max_steps and time.monotonic() - SESSION_START < RUN_CONFIG["qat_deadline_seconds"]:
        completed = resume["completed_qat_steps"]; session_epoch = completed // RUN_CONFIG["qat_steps_per_session_epoch"] + 1
        session_end = min(session_epoch * RUN_CONFIG["qat_steps_per_session_epoch"], max_steps)
        while resume["completed_qat_steps"] < session_end and time.monotonic() - SESSION_START < RUN_CONFIG["qat_deadline_seconds"]:
            mixture, clean = next(qat_iterator); qat_loss = float(qat_train_step(mixture, clean)); resume["completed_qat_steps"] += 1
            if resume["completed_qat_steps"] % PROGRESS_INTERVAL_STEPS == 0:
                resume["checkpoint"] = QAT_MANAGER.save(checkpoint_number=resume["completed_qat_steps"]); resume["last_loss"] = qat_loss; resume["elapsed_seconds"] = time.monotonic() - SESSION_START
                temporary = resume_path.with_suffix(".tmp"); temporary.write_text(json.dumps(resume, indent=2), encoding="utf-8"); temporary.replace(resume_path)
        completed = resume["completed_qat_steps"]; epoch, local_step = session_position(completed, RUN_CONFIG["qat_steps_per_session_epoch"])
        current = validation_loss(); qat_plateau.update(current)
        resume["checkpoint"] = QAT_MANAGER.save(checkpoint_number=completed); resume["last_loss"] = qat_loss; resume["elapsed_seconds"] = time.monotonic() - SESSION_START
        temporary = resume_path.with_suffix(".tmp"); temporary.write_text(json.dumps(resume, indent=2), encoding="utf-8"); temporary.replace(resume_path)
        elapsed = time.monotonic() - qat_clock; append_progress(RUN_DIR / "progress_qat.jsonl", {"phase":"qat", "global_step":completed, "session_epoch":epoch, "step_in_session_epoch":local_step, "loss":qat_loss, "validation_loss":current, "learning_rate":float(tf.keras.backend.get_value(qat_optimizer.learning_rate)), "seconds_per_step":elapsed / max(completed - qat_start, 1), "remaining_seconds":max(0.0, RUN_CONFIG["qat_deadline_seconds"] - (time.monotonic() - SESSION_START)), "target_steps":max_steps})
    resume["checkpoint"] = QAT_MANAGER.save(checkpoint_number=resume["completed_qat_steps"]); resume["last_loss"] = qat_loss; resume["training_budget_status"] = budget_status(resume["completed_qat_steps"], max_steps, RUN_CONFIG["qat_deadline_seconds"])
    resume_path.write_text(json.dumps(resume, indent=2), encoding="utf-8")
    FRAME_CORE.save_weights(RUN_DIR / "best_qat.weights.h5")
    (RUN_DIR / "qat_policy.json").write_text(json.dumps({**qat_policy, **resume}, indent=2), encoding="utf-8")
    return FRAME_CORE
if RUN_PROFILE == "p100_full_eval":
    QAT_ENABLED = True; prepare_qat_model(FRAME_CORE); FRAME_CORE.load_weights(FULL_EVAL_CHECKPOINT, skip_mismatch=True)
    QAT_METRICS = evaluate_speech_enhancement("qat", enhancement_fn=evaluation_enhance_sequence)
    QAT_ENABLED = False
    (RUN_DIR / "training_budget.json").write_text(json.dumps({"profile": RUN_PROFILE, "fp32": fp32_state, "qat": {"phase":"full_eval", "completed_qat_steps":0, "training_budget_status":"not_run"}, "training_budget_status":"not_run", "experimental_final_output":False, "release_ready":False}, indent=2), encoding="utf-8")
elif RUN_FINALIZATION:
    QAT_MODEL = run_qat(RUN_CONFIG["qat_max_steps"])
    # Trace this only after QAT is enabled so inference metrics use the trained fake-quant graph.
else:
    QAT_ENABLED = False; QAT_MODEL = FRAME_CORE; QAT_METRICS = None; QAT_EXPORT_ENABLED = False
    (RUN_DIR / "training_budget.json").write_text(json.dumps({"profile": RUN_PROFILE, "fp32": fp32_state, "qat": {"phase":"not_started", "completed_qat_steps":0, "training_budget_status":"not_run"}, "training_budget_status":fp32_state["training_budget_status"], "experimental_final_output":False, "release_ready":False}, indent=2), encoding="utf-8")
@tf.function(input_signature=(TRAIN_INPUT_SIGNATURE[0],), reduce_retracing=True)
def qat_enhance_sequence(waveform):
    with tf.device(GPU_TRAIN_DEVICE): return enhance_sequence(waveform)
@tf.function(input_signature=(tf.TensorSpec([None, None], tf.float32),), reduce_retracing=True)
def qat_evaluation_enhance_sequence(waveform):
    with tf.device(GPU_TRAIN_DEVICE): return enhance_sequence(waveform)
# Freeze the clean inference graph after the best QAT checkpoint; no training-only QAT switch is traced by the converter.
best_qat = RUN_DIR / "best_qat.weights.h5"
if RUN_FINALIZATION and RUN_PROFILE != "p100_full_eval" and best_qat.is_file(): FRAME_CORE.load_weights(best_qat)
if RUN_FINALIZATION and RUN_PROFILE != "p100_full_eval": QAT_METRICS = evaluate_speech_enhancement("qat", enhancement_fn=qat_evaluation_enhance_sequence)
QAT_EXPORT_ENABLED = RUN_FINALIZATION and RUN_PROFILE != "p100_full_eval"
# Export the QAT-trained FP32 master weights through TFLite's INT8 converter.
# Observers update moving variables during QAT; leaving them enabled would put
# AssignVariableOp resource edges inside the frozen inference graph.
if QAT_EXPORT_ENABLED:
    QAT_ENABLED = False; frozen_inference_model = FRAME_CORE
    FINAL_TFLITE = export_int8(PACKAGE / "neuralaid_int8.tflite", representative_frames(), frozen_inference_model)
    FINAL_AUDIT = vela_audit(FINAL_TFLITE, "final")
    (RUN_DIR / "qat_export_contract.json").write_text(json.dumps({"qat_mode": QAT_MODE, "qat_export_enabled": QAT_EXPORT_ENABLED, "activation": "per_site_moving_observer_asymmetric_lsq", "weights": "per_channel_symmetric_ste"}, indent=2), encoding="utf-8")
    (RUN_DIR / "training_budget.json").write_text(json.dumps({"profile": RUN_PROFILE, "fp32": fp32_state, "qat": json.loads((RUN_DIR / "qat_resume_state.json").read_text()), "training_budget_status": "completed" if fp32_state["training_budget_status"] == "completed" else fp32_state["training_budget_status"], "experimental_final_output": bool(RUN_CONFIG["final_in_session"]), "release_ready": False}, indent=2), encoding="utf-8")
    (RUN_DIR / "quantization_audit.json").write_text(json.dumps({"track": TRACK, "qat_mode": QAT_MODE, "qat_steps": RUN_CONFIG["qat_max_steps"], "status": "exported", "weight_quantization_coverage": prepare_qat_model(FRAME_CORE)["weight_quantization_coverage"]}, indent=2), encoding="utf-8")


## Firmware package

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Firmware package.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Firmware package; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
if RUN_FINALIZATION:
    def emit_model_cc(path):
        values = ", ".join(f"0x{b:02x}" for b in Path(path).read_bytes())
        (PACKAGE / "model_data.h").write_text("#pragma once\n#include <stddef.h>\nextern const unsigned char g_neuralaid_model[];\nextern const size_t g_neuralaid_model_len;\n", encoding="utf-8")
        (PACKAGE / "model_data.cc").write_text(f'#include "model_data.h"\nalignas(16) const unsigned char g_neuralaid_model[] = {{{values}}};\nconst size_t g_neuralaid_model_len = sizeof(g_neuralaid_model);\n', encoding="utf-8")
    raw_model = PACKAGE / "neuralaid_int8_raw.tflite"; shutil.copy2(FINAL_TFLITE, raw_model)
    selected = next((v["vela_tflite"] for v in FINAL_AUDIT["variants"] if v["cpu_operators_zero"] and v["vela_tflite"]), None)
    if selected:
        shutil.copy2(selected, PACKAGE / "neuralaid_int8_vela.tflite")
        emit_model_cc(PACKAGE / "neuralaid_int8_vela.tflite")
    elif TRACK == "u55_equivalent":
        (PACKAGE / "release_blocked.txt").write_text("U55 release blocked: no Vela artifact with CPU operators = 0\n", encoding="utf-8")
    (PACKAGE / "window_coeffs.h").write_text("#pragma once\nextern const float g_neuralaid_synthesis_window[160];\n", encoding="utf-8")
    (PACKAGE / "window_coeffs.c").write_text("#include \"window_coeffs.h\"\nconst float g_neuralaid_synthesis_window[160] = {" + ",".join(map(str, np.asarray(SYNTHESIS_WINDOW, np.float32))) + "};\n", encoding="utf-8")
    (PACKAGE / "neuralaid_contract.h").write_text("#pragma once\n#define NEURALAID_SAMPLE_RATE 16000\n#define NEURALAID_CHUNK_SAMPLES 96\n#define NEURALAID_LOOKBACK_SAMPLES 96\n#define NEURALAID_LOOKAHEAD_SAMPLES 64\n#define NEURALAID_FFT_LENGTH 256\n#define NEURALAID_FFT_BINS 129\n#define NEURALAID_LSTM_BLOCKS 6\n#define NEURALAID_MODEL_DIM 32\n#define NEURALAID_HIDDEN_SIZE 32\n#define NEURALAID_FREQ_COMPRESSION 4\n#define NEURALAID_PREEMPHASIS 0.97f\n#define NEURALAID_PREEMPHASIS_Q15 31785\n", encoding="utf-8")
else:
    print("run: skip conversion, Vela, golden and package generation")


## Firmware skeleton

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Firmware skeleton.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Firmware skeleton; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
if RUN_FINALIZATION:
    (PACKAGE / "neuralaid_pipeline.h").write_text("#pragma once\n#include <stdbool.h>\n#include <stddef.h>\n#include <stdint.h>\nvoid neuralaid_init(void*, size_t);\nvoid neuralaid_reset_stream(void);\nvoid neuralaid_push_audio(const int16_t*, size_t);\nbool neuralaid_pop_output(int16_t out96[96]);\n", encoding="utf-8")
    (PACKAGE / "neuralaid_pipeline.c").write_text('#include "neuralaid_pipeline.h"\n#include "neuralaid_contract.h"\n#include <limits.h>\nstatic int16_t audio_ring[NEURALAID_FFT_LENGTH], output_queue[4][NEURALAID_CHUNK_SAMPLES]; static unsigned write_index, pending, output_read, output_count; static int16_t previous_sample;\nstatic int16_t sat16(int32_t x){if(x>INT16_MAX)return INT16_MAX;if(x<INT16_MIN)return INT16_MIN;return (int16_t)x;}\nstatic void process_one_chunk(void){int16_t*out=output_queue[(output_read+output_count)%4];/* TODO production modules: context->CMSIS RFFT->pack/quantize->TFLM/Ethos-U->dequantize->iRFFT->window/OLA. */for(unsigned i=0;i<NEURALAID_CHUNK_SAMPLES;++i)out[i]=0;if(output_count<4)++output_count;}\nvoid neuralaid_reset_stream(void){for(unsigned i=0;i<NEURALAID_FFT_LENGTH;++i)audio_ring[i]=0;write_index=pending=output_read=output_count=0;previous_sample=0;}\nvoid neuralaid_init(void*a,size_t n){(void)a;(void)n;neuralaid_reset_stream();}\nvoid neuralaid_push_audio(const int16_t*in,size_t count){for(size_t i=0;i<count;++i){int32_t y=(int32_t)in[i]-((NEURALAID_PREEMPHASIS_Q15*(int32_t)previous_sample)>>15);previous_sample=in[i];audio_ring[write_index]=sat16(y);write_index=(write_index+1u)%NEURALAID_FFT_LENGTH;++pending;while(pending>=NEURALAID_CHUNK_SAMPLES+NEURALAID_LOOKAHEAD_SAMPLES&&output_count<4){pending-=NEURALAID_CHUNK_SAMPLES;process_one_chunk();}}}\nbool neuralaid_pop_output(int16_t out96[96]){if(!output_count)return false;for(unsigned i=0;i<NEURALAID_CHUNK_SAMPLES;++i)out96[i]=output_queue[output_read][i];output_read=(output_read+1u)%4;--output_count;return true;}\n', encoding="utf-8")
else:
    print("run: skip conversion, Vela, golden and package generation")


## Golden and manifest

**Mục tiêu:** Thực hiện riêng nhiệm vụ: Golden and manifest.

**Đầu vào:** Tensor, cấu hình hoặc artefact của cell ngay trước.

**Đầu ra:** Biến runtime hoặc artefact phục vụ cell tiếp theo.

**Quyết định kỹ thuật:** Cell này chỉ phụ trách Golden and manifest; không gộp nhiệm vụ khác.

**Artifact sinh ra:** Được ghi rõ trong code của cell kế tiếp.


In [ ]:
if RUN_FINALIZATION:
    STREAMING_GOLDEN_CHUNKS = 100
    def quantize_state(real, scale, zero_point): return np.clip(np.rint(np.asarray(real, np.float32) / scale + zero_point), -128, 127).astype(np.int8)
    def dequantize_state(encoded, scale, zero_point): return np.asarray(encoded, np.float32) if scale == 0 else (np.asarray(encoded, np.float32) - zero_point) * scale
    def run_raw_int8_stream(interpreter, tensor_map, output_map, contexts):
        """Streaming oracle: outputs are fed back only through explicit state quantization pairs."""
        state = {name: np.zeros(detail["shape"], np.float32) for name, detail in tensor_map.items() if name != "tf_frame"}; trace = []
        for index in range(min(STREAMING_GOLDEN_CHUNKS, int(tf.shape(contexts)[1]))):
            spectrum = frontend_frame(contexts[:, index, :]); values = {"tf_frame": tf.stack([tf.math.real(spectrum), tf.math.imag(spectrum)], -1).numpy(), **state}
            for name, detail in tensor_map.items():
                scale, zero = detail["quantization"]; interpreter.set_tensor(detail["index"], quantize_state(values[name], scale, zero) if scale else values[name].astype(detail["dtype"]))
            interpreter.invoke(); outputs = {detail["name"]: interpreter.get_tensor(detail["index"]).copy() for detail in interpreter.get_output_details()}
            for name, detail in output_map.items():
                encoded = interpreter.get_tensor(detail["index"]); scale, zero = detail["quantization"]; state[name] = dequantize_state(encoded, scale, zero)
            trace.append({"chunk": index, "outputs": outputs, "state": {name: value.copy() for name, value in state.items()}})
        return trace
    golden = PACKAGE / "golden"; golden.mkdir(exist_ok=True); stages = golden / "stages"; stages.mkdir(exist_ok=True); t = np.arange(CHUNK_SAMPLES * STREAMING_GOLDEN_CHUNKS); signals = {"silence": np.zeros_like(t,np.float32), "impulse": np.r_[1.,np.zeros(t.size-1,np.float32)], "sine_440": np.sin(2*np.pi*440*t/SAMPLE_RATE).astype(np.float32), "sine_3000": np.sin(2*np.pi*3000*t/SAMPLE_RATE).astype(np.float32), "noise": np.random.default_rng(SEED).normal(size=t.size).astype(np.float32)}
    for name, signal in signals.items():
        np.save(golden / f"{name}.npy", signal)
        waveform = tf.convert_to_tensor(signal[None, :], tf.float32); pre = pre_emphasis_sequence(waveform); np.save(stages / f"{name}_preemphasis.npy", pre.numpy())
        contexts = make_stream_contexts(waveform); np.save(stages / f"{name}_context_000.npy", contexts[:, 0].numpy()); spectrum = frontend_frame(contexts[:, 0]); np.save(stages / f"{name}_fft_real_000.npy", tf.math.real(spectrum).numpy()); np.save(stages / f"{name}_fft_imag_000.npy", tf.math.imag(spectrum).numpy())
        state = initial_state(1); result = FRAME_CORE({"tf_frame": tf.stack([tf.math.real(spectrum), tf.math.imag(spectrum)], -1), **state})
        for key, value in state.items(): np.save(stages / f"{name}_state_in_000_{key}.npy", value.numpy())
        for key in state: np.save(stages / f"{name}_state_out_000_{key}.npy", result[f"next_{key}"].numpy())
        model_tf = tf.complex(result["enhanced_tf"][..., 0], result["enhanced_tf"][..., 1]); np.save(stages / f"{name}_model_output_000.npy", result["enhanced_tf"].numpy()); np.save(stages / f"{name}_ifft_000.npy", tf.signal.irfft(model_tf, [FFT_LENGTH]).numpy()); np.save(stages / f"{name}_synthesis_000.npy", dual_window_istft(model_tf).numpy())
    state_contract = {"track": TRACK, "inputs": list(initial_state()) + ["tf_frame"], "state_shapes": {k:list(v) for k,v in state_shapes(1).items()}, "state_reset": "zero_on_boot_or_stream_discontinuity", "dataset": DATASET_PROVENANCE}
    HOST_GOLDEN_MODEL = PACKAGE / "neuralaid_int8_raw.tflite"
    DEPLOY_MODEL = PACKAGE / "neuralaid_int8_vela.tflite"
    if HOST_GOLDEN_MODEL.is_file():
        interpreter = tf.lite.Interpreter(model_path=str(HOST_GOLDEN_MODEL)); interpreter.allocate_tensors()
        state_contract["tflite_inputs"] = [{"index":int(x["index"]),"name":x["name"],"shape":x["shape"].tolist(),"dtype":str(x["dtype"]),"quantization":list(x["quantization"])} for x in interpreter.get_input_details()]
        state_contract["tflite_outputs"] = [{"index":int(x["index"]),"name":x["name"],"shape":x["shape"].tolist(),"dtype":str(x["dtype"]),"quantization":list(x["quantization"])} for x in interpreter.get_output_details()]
        # Golden tensors are emitted from the actual raw-or-Vela deployed graph.
        logical_inputs = ["tf_frame", *initial_state(1)]
        details_by_name = {detail["name"].split(":")[0]: detail for detail in interpreter.get_input_details()}
        next_details_by_name = {detail["name"].split(":")[0]: detail for detail in interpreter.get_output_details()}
        LOGICAL_TENSOR_MAP = {name: next((detail for tensor_name, detail in details_by_name.items() if tensor_name == name or tensor_name.endswith("_" + name)), None) for name in logical_inputs}
        if any(detail is None for detail in LOGICAL_TENSOR_MAP.values()): raise RuntimeError("unmapped logical TFLite input; export signature names required")
        LOGICAL_OUTPUT_MAP = {name: next((detail for tensor_name, detail in next_details_by_name.items() if tensor_name == "next_" + name or tensor_name.endswith("_next_" + name)), None) for name in initial_state(1)}
        if any(detail is None for detail in LOGICAL_OUTPUT_MAP.values()):
            # TFLite may erase dictionary output names; fallback to exported output order.
            output_details = interpreter.get_output_details()
            expected_output_order = sorted(["enhanced_tf", *["next_" + name for name in initial_state(1)]])
            # Also repair a live notebook session whose prior export recorded the
            # pre-fix insertion order; no FP32/QAT rerun is required for Golden.
            if EXPORT_LOGICAL_OUTPUT_ORDER != expected_output_order:
                EXPORT_LOGICAL_OUTPUT_ORDER = expected_output_order
            if len(EXPORT_LOGICAL_OUTPUT_ORDER) != len(output_details):
                raise RuntimeError({"logical_outputs": EXPORT_LOGICAL_OUTPUT_ORDER, "tflite_outputs": [detail["name"] for detail in output_details]})
            output_by_logical_name = dict(zip(EXPORT_LOGICAL_OUTPUT_ORDER, interpreter.get_output_details()))
            LOGICAL_OUTPUT_MAP = {name: output_by_logical_name.get("next_" + name) for name in initial_state(1)}
            if any(detail is None for detail in LOGICAL_OUTPUT_MAP.values()):
                raise RuntimeError({"missing_state_outputs": [name for name, detail in LOGICAL_OUTPUT_MAP.items() if detail is None], "logical_outputs": EXPORT_LOGICAL_OUTPUT_ORDER})
            for name, detail in LOGICAL_OUTPUT_MAP.items():
                if tuple(detail["shape"]) != tuple(state_shapes(1)[name]):
                    raise RuntimeError({"state_output_shape_mismatch": name, "expected": list(state_shapes(1)[name]), "actual": detail["shape"].tolist()})
            state_contract["tflite_output_mapping"] = "export_logical_output_order_fallback"
        state_pairs = [{"input_name": name, "input_index": int(detail["index"]), "input_shape": detail["shape"].tolist(), "input_dtype": str(detail["dtype"]), "input_scale": detail["quantization"][0], "input_zero_point": detail["quantization"][1], "output_name": "next_" + name, "output_index": int(LOGICAL_OUTPUT_MAP[name]["index"]), "output_shape": LOGICAL_OUTPUT_MAP[name]["shape"].tolist(), "output_dtype": str(LOGICAL_OUTPUT_MAP[name]["dtype"]), "output_scale": LOGICAL_OUTPUT_MAP[name]["quantization"][0], "output_zero_point": LOGICAL_OUTPUT_MAP[name]["quantization"][1], "raw_copy_ok": detail["quantization"] == LOGICAL_OUTPUT_MAP[name]["quantization"], "requant_policy": "dequantize_then_requantize"} for name, detail in LOGICAL_TENSOR_MAP.items() if name != "tf_frame"]
        (PACKAGE / "state_quant_contract.json").write_text(json.dumps({"state_pairs": state_pairs}, indent=2), encoding="utf-8")
        probe = tf.sin(tf.range(FFT_LENGTH, dtype=tf.float32)[None, :] * 0.07); probe_frame = frontend_frame(make_stream_contexts(probe)[:, 0, :]); probe_values = {"tf_frame": tf.stack([tf.math.real(probe_frame), tf.math.imag(probe_frame)], -1).numpy(), **{k:v.numpy() for k,v in initial_state(1).items()}}
        for logical_name, detail in LOGICAL_TENSOR_MAP.items():
            source = probe_values[logical_name]
            scale, zero = detail["quantization"]; encoded = np.asarray(source, np.float32) if scale == 0 else np.clip(np.rint(source / scale + zero), -128, 127).astype(np.int8)
            interpreter.set_tensor(detail["index"], encoded); np.save(stages / f"deployed_input_{detail['index']}.npy", encoded)
        interpreter.invoke()
        for detail in interpreter.get_output_details():
            encoded = interpreter.get_tensor(detail["index"]); scale, zero = detail["quantization"]; decoded = encoded.astype(np.float32) if scale == 0 else (encoded.astype(np.float32) - zero) * scale
            np.save(stages / f"deployed_output_int8_{detail['index']}.npy", encoded); np.save(stages / f"deployed_output_dequantized_{detail['index']}.npy", decoded)
        for name, signal in signals.items(): np.save(stages / f"{name}_raw_int8_stream_trace.npy", np.asarray(run_raw_int8_stream(interpreter, LOGICAL_TENSOR_MAP, LOGICAL_OUTPUT_MAP, make_stream_contexts(tf.convert_to_tensor(signal[None, :], tf.float32))), dtype=object), allow_pickle=True)
    (PACKAGE / "tensor_contract.json").write_text(json.dumps(state_contract, indent=2), encoding="utf-8")
    release = {"track":TRACK,"vela_exit_ok":all(v["exit_status"] == 0 for v in FINAL_AUDIT["variants"]),"cpu_operators_zero":FINAL_AUDIT["deploy_ready"],"functional_parity_pass":False,"perfect_reconstruction_pass":False,"sram_budget_pass":False,"flash_budget_pass":False,"runtime_deadline_pass":False,"training_budget_status":json.loads((RUN_DIR / "training_budget.json").read_text())["training_budget_status"],"experimental_final_output":bool(RUN_CONFIG["final_in_session"]),"release_ready":False}
    (PACKAGE / "release_gates.json").write_text(json.dumps(release,indent=2),encoding="utf-8")
    manifest = {"track":TRACK,"experiment_id":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"dataset":"speech_enhancement_board_response_v1","dataset_provenance":DATASET_PROVENANCE,"training_budget":json.loads((RUN_DIR / "training_budget.json").read_text()),"preflight":PREFLIGHT_AUDIT,"final":FINAL_AUDIT,"deploy_status":FINAL_AUDIT["deployment_status"],"raw_model":str(PACKAGE/"neuralaid_int8_raw.tflite"),"sha256":hashlib.sha256((PACKAGE/"neuralaid_int8_raw.tflite").read_bytes()).hexdigest()}
    if (PACKAGE / "neuralaid_int8_vela.tflite").is_file(): manifest["vela_model"] = str(PACKAGE / "neuralaid_int8_vela.tflite")
    (PACKAGE / "integration_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
else:
    print("run: skip conversion, Vela, golden and package generation")


## Track parity and release gates

**Mục tiêu:** Track parity and release gates.



**Artifact sinh ra:** được ghi rõ trong code của cell này.


In [ ]:
if RUN_FINALIZATION:
    def frontend_state_contract(reference, candidate):
        assert set(initial_state()) == set(state_shapes())
        assert tuple(reference.input_shape[1:]) == tuple(candidate.input_shape[1:])
        return {"state_names": sorted(initial_state()), "input_shape": [1, FFT_BINS, 2]}

    def compare_track_task_metrics(metrics_a, metrics_b, max_si_sdri_drop=0.5, max_pesq_drop=0.1):
        delta = {"si_sdri": metrics_b["si_sdri"] - metrics_a["si_sdri"], "pesq": metrics_b["pesq"] - metrics_a["pesq"]}
        passed = delta["si_sdri"] >= -max_si_sdri_drop and delta["pesq"] >= -max_pesq_drop
        return {"passed": passed, "delta": delta, "thresholds": {"si_sdri": max_si_sdri_drop, "pesq": max_pesq_drop}}

    def streaming_dsp_identity_test():
        probe = tf.sin(tf.range(768, dtype=tf.float32)[None, :] * 0.07)
        expected = pre_emphasis_sequence(probe)
        reconstruction = initial_reconstruction_state(1); chunks = []; contexts = make_stream_contexts(probe)
        for index in range(int(tf.shape(contexts)[1].numpy())):
            short = normalized_reference_synthesis(frontend_frame(contexts[:, index, :]))
            emitted, reconstruction["ola_tail"], reconstruction["ola_weight_tail"] = synthesis_stream(short, reconstruction["ola_tail"], reconstruction["ola_weight_tail"]); chunks.append(emitted)
        reconstructed = tf.concat(chunks, axis=1)[:, :tf.shape(expected)[1]]
        error = float(tf.reduce_max(tf.abs(expected - reconstructed)))
        if error >= 1e-5: raise AssertionError({"identity_reconstruction_max_error": error})
        return error

    STREAMING_DSP_IDENTITY_MAX_ERROR = streaming_dsp_identity_test()

    def load_parity_metrics(reference_metrics_path, candidate_metrics_path, max_tf_mse=1e-3, max_waveform_mse=1e-3, max_si_sdri_drop=0.5, max_pesq_drop=0.1):
        reference, candidate = json.loads(Path(reference_metrics_path).read_text()), json.loads(Path(candidate_metrics_path).read_text())
        result = {"tf_mse": candidate["tf_mse"], "waveform_mse": candidate["waveform_mse"], "si_sdri_delta": candidate["si_sdri"] - reference["si_sdri"], "pesq_delta": candidate["pesq"] - reference["pesq"]}
        result["passed"] = result["tf_mse"] <= max_tf_mse and result["waveform_mse"] <= max_waveform_mse and result["si_sdri_delta"] >= -max_si_sdri_drop and result["pesq_delta"] >= -max_pesq_drop
        return result

    release_path = PACKAGE / "release_gates.json"; release = json.loads(release_path.read_text())
    release["perfect_reconstruction_pass"] = STREAMING_DSP_IDENTITY_MAX_ERROR < 1e-5
    release["functional_parity_pass"] = False  # Set only by measured Track A/Track B metrics after train/distill.
    release["streaming_dsp_identity_max_error"] = STREAMING_DSP_IDENTITY_MAX_ERROR
    release_path.write_text(json.dumps(release, indent=2), encoding="utf-8")

    if TRACK == "u55_equivalent":
        print("Track B release requires measured Track A/Track B TF, waveform, SI-SDRi and PESQ parity")
else:
    print("run: skip conversion, Vela, golden and package generation")
